In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math
import numpy as np

In [2]:
df = pd.read_csv(
   "../../data/processed/FullDataSet.csv"
)
df.head()

,Date,Time,Category,Item,Qty,Price Point Name,SKU,Unit Price,Gross Sales,Discounts,...,Gross Profit,Weekday,Event,Season x Month x Event Mult,DOW Mult,Month Mult,Event Mult,Discount Line Prob,Refund Txn Prob,Store Closed
0,2024-01-01,14:41:00,Home Goods,Ceramic Coffee Mug,1,Regular,SQ-1058,16.0,16.0,0.0,...,11.52,Monday,New Year's Day,0.401,0.9,0.8,0.5,0.3,0.015,0
1,2024-01-01,15:50:00,Apparel,Classic Denim Jacket,1,S,SQ-1001,79.0,79.0,0.0,...,43.45,Monday,New Year's Day,0.401,0.9,0.8,0.5,0.3,0.015,0
2,2024-01-01,13:11:00,Electronics,Wireless Earbuds,1,Black,SQ-1048,79.0,79.0,0.0,...,35.55,Monday,New Year's Day,0.401,0.9,0.8,0.5,0.3,0.015,0
3,2024-01-01,15:33:00,Apparel,Hooded Sweatshirt,2,M,SQ-1013,45.0,90.0,-13.5,...,40.50,Monday,New Year's Day,0.401,0.9,0.8,0.5,0.3,0.015,0
4,2024-01-01,15:33:00,Footwear,Canvas Sneakers,1,10,SQ-1027,59.0,59.0,-11.8,...,18.88,Monday,New Year's Day,0.401,0.9,0.8,0.5,0.3,0.015,0


In [3]:
df["Total_Cost"] = df["Unit Cost"] * df["Qty"]
df["Is_Refund"] = (df["Event Type"] == "Refund").astype(int)
df["Has_Discount"] = (df["Discounts"] < 0).astype(int)
df["Has_Customer"] = df["Customer ID"].notna().astype(int)

In [4]:
store_day = (
    df
    .groupby(["Location", "Date"])
    .agg(
        Gross_Sales=("Gross Sales", "sum"),
        Discounts=("Discounts", "sum"),
        Net_Sales=("Net Sales", "sum"),
        Tax=("Tax", "sum"),
        Total_Collected=("Total Collected", "sum"),
        Gross_Profit=("Gross Profit", "sum"),
        Total_Cost=("Total_Cost", "sum"),
        Transactions=("Transaction ID", "nunique"),
        Units=("Qty", "sum"),
        Refund_Lines=("Is_Refund", "sum"),
        Discounted_Lines=("Has_Discount", "sum"),
        Identified_Customer_Lines=("Has_Customer", "sum"),
        Unique_SKUs=("SKU", "nunique"),
        Weekday=("Weekday", "first"),
        Event=("Event", "first"),
        Season_Month_Event_Mult=("Season x Month x Event Mult", "first"),
        DOW_Mult=("DOW Mult", "first"),
        Month_Mult=("Month Mult", "first"),
        Event_Mult=("Event Mult", "first"),
        Discount_Line_Prob=("Discount Line Prob", "first"),
        Refund_Txn_Prob=("Refund Txn Prob", "first"),
        Store_Closed=("Store Closed", "first")
    )
    .reset_index()
)

In [5]:
store_day.shape

(21862, 24)

In [6]:
store_day.head()


,Location,Date,Gross_Sales,Discounts,Net_Sales,Tax,Total_Collected,Gross_Profit,Total_Cost,Transactions,...,Unique_SKUs,Weekday,Event,Season_Month_Event_Mult,DOW_Mult,Month_Mult,Event_Mult,Discount_Line_Prob,Refund_Txn_Prob,Store_Closed
0,Store 01 - Austin,2024-01-01,456.0,-37.15,418.85,34.56,453.41,212.82,206.03,5,...,7,Monday,New Year's Day,0.401,0.90,0.8,0.50,0.3,0.0150,0
1,Store 01 - Austin,2024-01-02,238.0,-5.90,232.10,19.15,251.25,130.46,101.64,4,...,5,Tuesday,January Returns Tail,0.682,0.85,0.8,0.85,0.3,0.0375,0
2,Store 01 - Austin,2024-01-03,296.0,-40.72,255.28,21.07,276.35,136.06,119.22,2,...,7,Wednesday,January Returns Tail,0.683,0.90,0.8,0.85,0.3,0.0375,0
3,Store 01 - Austin,2024-01-04,478.0,-12.54,465.46,38.42,503.88,281.20,184.26,9,...,16,Thursday,January Returns Tail,0.684,1.00,0.8,0.85,0.3,0.0375,0
4,Store 01 - Austin,2024-01-05,902.0,-59.07,842.93,69.54,912.47,487.54,355.39,16,...,23,Friday,January Returns Tail,0.685,1.25,0.8,0.85,0.3,0.0375,0


In [7]:
store_day.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21862 entries, 0 to 21861
Data columns (total 24 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Location                   21862 non-null  object 
 1   Date                       21862 non-null  object 
 2   Gross_Sales                21862 non-null  float64
 3   Discounts                  21862 non-null  float64
 4   Net_Sales                  21862 non-null  float64
 5   Tax                        21862 non-null  float64
 6   Total_Collected            21862 non-null  float64
 7   Gross_Profit               21862 non-null  float64
 8   Total_Cost                 21862 non-null  float64
 9   Transactions               21862 non-null  int64  
 10  Units                      21862 non-null  int64  
 11  Refund_Lines               21862 non-null  int32  
 12  Discounted_Lines           21862 non-null  int32  
 13  Identified_Customer_Lines  21862 non-null  int

In [8]:
all_dates = pd.date_range(
    start=store_day["Date"].min(),
    end=store_day["Date"].max(),
    freq="D"
)

all_stores = store_day["Location"].unique()

complete_calendar = pd.MultiIndex.from_product(
    [all_stores, all_dates],
    names=["Location", "Date"]
).to_frame(index=False)

In [9]:
store_day['Date'] = pd.to_datetime(store_day['Date'])

In [10]:
df['Date'] = pd.to_datetime(df['Date'])

In [11]:
store_day = complete_calendar.merge(
    store_day,
    on=["Location", "Date"],
    how="left"
)

In [12]:
missing_store_days = store_day[
    store_day["Gross_Sales"].isna()
][["Location", "Date"]]

missing_store_days

,Location,Date
359,Store 01 - Austin,2024-12-25
696,Store 01 - Austin,2025-11-27
724,Store 01 - Austin,2025-12-25
1063,Store 02 - Dallas,2024-11-28
1090,Store 02 - Dallas,2024-12-25
...,...,...
20461,Store 28 - Sacramento,2025-12-25
20827,Store 29 - Minneapolis,2024-12-25
21192,Store 29 - Minneapolis,2025-12-25
21558,Store 30 - Kansas City,2024-12-25


In [13]:
missing_store_days["Date"].value_counts().sort_index()

Date
2024-11-28     5
2024-12-25    30
2025-01-01     1
2025-11-27     2
2025-12-25    30
Name: count, dtype: int64

In [14]:
zero_cols = [
    "Gross_Sales",
    "Discounts",
    "Net_Sales",
    "Tax",
    "Total_Collected",
    "Gross_Profit",
    "Total_Cost",
    "Transactions",
    "Units",
    "Refund_Lines",
    "Discounted_Lines",
    "Identified_Customer_Lines",
    "Unique_SKUs"
]

closed_mask = store_day["Store_Closed"].isna()

store_day.loc[closed_mask, zero_cols] = 0
store_day.loc[closed_mask, "Store_Closed"] = 1

In [15]:
calendar_cols = [
    "Date",
    "Weekday",
    "Event",
    "Season x Month x Event Mult",
    "DOW Mult",
    "Month Mult",
    "Event Mult",
    "Discount Line Prob",
    "Refund Txn Prob"
]

calendar = df[calendar_cols].drop_duplicates("Date").copy()

# Christmas has no transaction rows, so add its calendar information manually
# from the ground-truth event calendar.
christmas = pd.DataFrame({
    "Date": pd.to_datetime(["2024-12-25", "2025-12-25"]),
    "Weekday": ["Wednesday", "Thursday"],
    "Event": ["Christmas Day (closed)", "Christmas Day (closed)"],
    "Season x Month x Event Mult": [0.0, 0.0],
    "DOW Mult": [0.9, 1.0],
    "Month Mult": [1.6, 1.6],
    "Event Mult": [0.0, 0.0],
    "Discount Line Prob": [0.4, 0.4],
    "Refund Txn Prob": [0.015, 0.015]
})

calendar = pd.concat([calendar, christmas], ignore_index=True)
calendar = calendar.drop_duplicates("Date", keep="last")

store_day = store_day.drop(
    columns=[
        "Weekday",
        "Event",
        "Season_Month_Event_Mult",
        "DOW_Mult",
        "Month_Mult",
        "Event_Mult",
        "Discount_Line_Prob",
        "Refund_Txn_Prob"
    ]
)

In [16]:
store_day = store_day.merge(
    calendar,
    on="Date",
    how="left"
)

In [17]:
store_day.rename(columns={
    "Season x Month x Event Mult": "Season_Month_Event_Mult",
    "DOW Mult": "DOW_Mult",
    "Month Mult": "Month_Mult",
    "Event Mult": "Event_Mult",
    "Discount Line Prob": "Discount_Line_Prob",
    "Refund Txn Prob": "Refund_Txn_Prob"
}, inplace=True)

In [18]:
store_day.isna().sum()

Location                     0
Date                         0
Gross_Sales                  0
Discounts                    0
Net_Sales                    0
Tax                          0
Total_Collected              0
Gross_Profit                 0
Total_Cost                   0
Transactions                 0
Units                        0
Refund_Lines                 0
Discounted_Lines             0
Identified_Customer_Lines    0
Unique_SKUs                  0
Store_Closed                 0
Weekday                      0
Event                        0
Season_Month_Event_Mult      0
DOW_Mult                     0
Month_Mult                   0
Event_Mult                   0
Discount_Line_Prob           0
Refund_Txn_Prob              0
dtype: int64

In [19]:
store_day["Profit_Margin"] = (
    store_day["Gross_Profit"] / store_day["Net_Sales"]
).replace([np.inf, -np.inf], np.nan)

store_day["Discount_Rate"] = (
    -store_day["Discounts"] / store_day["Gross_Sales"]
).replace([np.inf, -np.inf], np.nan)

store_day["Avg_Transaction_Value"] = (
    store_day["Net_Sales"] / store_day["Transactions"]
).replace([np.inf, -np.inf], np.nan)

store_day["Units_Per_Transaction"] = (
    store_day["Units"] / store_day["Transactions"]
).replace([np.inf, -np.inf], np.nan)

In [20]:
store_day.describe().T

,count,mean,min,25%,50%,75%,max,std
Date,21930,2024-12-30 23:59:59.999999744,2024-01-01 00:00:00,2024-07-01 00:00:00,2024-12-31 00:00:00,2025-07-02 00:00:00,2025-12-31 00:00:00,NaN
Gross_Sales,21930.0,2246.671249,0.0,1209.0,1877.0,2882.75,22255.0,1535.755951
Discounts,21930.0,-121.948346,-2867.55,-143.3975,-87.15,-50.35,0.0,158.285319
Net_Sales,21930.0,2124.722903,0.0,1151.0625,1784.175,2740.43,19829.05,1414.741526
Tax,21930.0,175.304005,0.0,94.955,147.2,226.1175,1636.02,116.724908
Total_Collected,21930.0,2300.026908,0.0,1246.0175,1931.385,2966.5425,21465.07,1531.466432
Gross_Profit,21930.0,1175.743563,0.0,644.62,992.245,1513.5475,10303.56,764.293525
Total_Cost,21930.0,948.97934,0.0,505.39,792.755,1223.565,9525.49,655.094279
Transactions,21930.0,26.707296,0.0,15.0,22.0,34.0,266.0,17.854761
Units,21930.0,53.493662,0.0,29.0,45.0,68.0,525.0,36.053433


In [21]:
store_day

,Location,Date,Gross_Sales,Discounts,Net_Sales,Tax,Total_Collected,Gross_Profit,Total_Cost,Transactions,...,Season_Month_Event_Mult,DOW_Mult,Month_Mult,Event_Mult,Discount_Line_Prob,Refund_Txn_Prob,Profit_Margin,Discount_Rate,Avg_Transaction_Value,Units_Per_Transaction
0,Store 01 - Austin,2024-01-01,456.0,-37.15,418.85,34.56,453.41,212.82,206.03,5.0,...,0.401,0.90,0.8,0.50,0.3,0.0150,0.508106,0.081469,83.770000,1.600000
1,Store 01 - Austin,2024-01-02,238.0,-5.90,232.10,19.15,251.25,130.46,101.64,4.0,...,0.682,0.85,0.8,0.85,0.3,0.0375,0.562085,0.024790,58.025000,1.500000
2,Store 01 - Austin,2024-01-03,296.0,-40.72,255.28,21.07,276.35,136.06,119.22,2.0,...,0.683,0.90,0.8,0.85,0.3,0.0375,0.532983,0.137568,127.640000,4.000000
3,Store 01 - Austin,2024-01-04,478.0,-12.54,465.46,38.42,503.88,281.20,184.26,9.0,...,0.684,1.00,0.8,0.85,0.3,0.0375,0.604134,0.026234,51.717778,1.888889
4,Store 01 - Austin,2024-01-05,902.0,-59.07,842.93,69.54,912.47,487.54,355.39,16.0,...,0.685,1.25,0.8,0.85,0.3,0.0375,0.578387,0.065488,52.683125,1.312500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21925,Store 30 - Kansas City,2025-12-27,8676.0,-1355.31,7320.69,603.98,7924.67,3468.03,3852.66,99.0,...,2.147,1.55,1.6,1.35,0.5,0.0525,0.473730,0.156214,73.946364,1.888889
21926,Store 30 - Kansas City,2025-12-28,7052.5,-1034.28,6018.22,496.56,6514.78,3036.21,2982.01,81.0,...,2.150,1.35,1.6,1.35,0.5,0.0525,0.504503,0.146654,74.299012,2.111111
21927,Store 30 - Kansas City,2025-12-29,3322.0,-525.95,2796.05,230.69,3026.74,1401.85,1394.20,40.0,...,2.153,0.90,1.6,1.35,0.5,0.0525,0.501368,0.158323,69.901250,2.150000
21928,Store 30 - Kansas City,2025-12-30,4798.5,-761.88,4036.62,333.07,4369.69,1986.27,2050.35,47.0,...,2.156,0.85,1.6,1.35,0.5,0.0525,0.492063,0.158775,85.885532,2.319149


In [22]:
store_day["Profit_Lag_1"] = (
    store_day.groupby("Location")["Gross_Profit"]
    .shift(1)
)
store_day["Profit_Lag_7"] = (
    store_day.groupby("Location")["Gross_Profit"]
    .shift(7)
)
store_day["Net_Sales_Lag_1"] = (
    store_day.groupby("Location")["Net_Sales"]
    .shift(1)
)

store_day["Net_Sales_Lag_7"] = (
    store_day.groupby("Location")["Net_Sales"]
    .shift(7)
)

store_day["Transactions_Lag_1"] = (
    store_day.groupby("Location")["Transactions"]
    .shift(1)
)

store_day["Transactions_Lag_7"] = (
    store_day.groupby("Location")["Transactions"]
    .shift(7)
)

In [23]:
store_day[
    ["Location", "Date", "Gross_Profit", "Profit_Lag_1", "Profit_Lag_7", "Net_Sales_Lag_1", "Net_Sales_Lag_7", "Transactions_Lag_1", "Transactions_Lag_7"]
].head(10)

,Location,Date,Gross_Profit,Profit_Lag_1,Profit_Lag_7,Net_Sales_Lag_1,Net_Sales_Lag_7,Transactions_Lag_1,Transactions_Lag_7
0,Store 01 - Austin,2024-01-01,212.82,NaN,NaN,NaN,NaN,NaN,NaN
1,Store 01 - Austin,2024-01-02,130.46,212.82,NaN,418.85,NaN,5.0,NaN
2,Store 01 - Austin,2024-01-03,136.06,130.46,NaN,232.10,NaN,4.0,NaN
3,Store 01 - Austin,2024-01-04,281.20,136.06,NaN,255.28,NaN,2.0,NaN
4,Store 01 - Austin,2024-01-05,487.54,281.20,NaN,465.46,NaN,9.0,NaN
5,Store 01 - Austin,2024-01-06,461.39,487.54,NaN,842.93,NaN,16.0,NaN
6,Store 01 - Austin,2024-01-07,695.99,461.39,NaN,808.55,NaN,11.0,NaN
7,Store 01 - Austin,2024-01-08,588.44,695.99,212.82,1223.58,418.85,14.0,5.0
8,Store 01 - Austin,2024-01-09,342.17,588.44,130.46,984.10,232.10,12.0,4.0
9,Store 01 - Austin,2024-01-10,478.50,342.17,136.06,574.65,255.28,10.0,2.0


In [24]:
store_day["Profit_Rolling_7D"] = (
    store_day.groupby("Location")["Gross_Profit"]
    .transform(lambda x: x.rolling(7).mean())
)
store_day["Net_Sales_Rolling_7D"] = (
    store_day.groupby("Location")["Net_Sales"]
    .transform(lambda x: x.rolling(7).mean())
)
store_day["Transactions_Rolling_7D"] = (
    store_day.groupby("Location")["Transactions"]
    .transform(lambda x: x.rolling(7).mean())
)

In [25]:
store_day["Profit_Growth_7D"] = (
    (store_day["Gross_Profit"] - store_day["Profit_Lag_7"])
    / store_day["Profit_Lag_7"]
).replace([np.inf, -np.inf], np.nan)

store_day["Net_Sales_Growth_7D"] = (
    (store_day["Net_Sales"] - store_day["Net_Sales_Lag_7"])
    / store_day["Net_Sales_Lag_7"]
).replace([np.inf, -np.inf], np.nan)

store_day["Transactions_Growth_7D"] = (
    (store_day["Transactions"] - store_day["Transactions_Lag_7"])
    / store_day["Transactions_Lag_7"]
).replace([np.inf, -np.inf], np.nan)

In [26]:
store_day.columns

Index(['Location', 'Date', 'Gross_Sales', 'Discounts', 'Net_Sales', 'Tax',
       'Total_Collected', 'Gross_Profit', 'Total_Cost', 'Transactions',
       'Units', 'Refund_Lines', 'Discounted_Lines',
       'Identified_Customer_Lines', 'Unique_SKUs', 'Store_Closed', 'Weekday',
       'Event', 'Season_Month_Event_Mult', 'DOW_Mult', 'Month_Mult',
       'Event_Mult', 'Discount_Line_Prob', 'Refund_Txn_Prob', 'Profit_Margin',
       'Discount_Rate', 'Avg_Transaction_Value', 'Units_Per_Transaction',
       'Profit_Lag_1', 'Profit_Lag_7', 'Net_Sales_Lag_1', 'Net_Sales_Lag_7',
       'Transactions_Lag_1', 'Transactions_Lag_7', 'Profit_Rolling_7D',
       'Net_Sales_Rolling_7D', 'Transactions_Rolling_7D', 'Profit_Growth_7D',
       'Net_Sales_Growth_7D', 'Transactions_Growth_7D'],
      dtype='object')

In [27]:
store_day.isna().sum()

Location                       0
Date                           0
Gross_Sales                    0
Discounts                      0
Net_Sales                      0
Tax                            0
Total_Collected                0
Gross_Profit                   0
Total_Cost                     0
Transactions                   0
Units                          0
Refund_Lines                   0
Discounted_Lines               0
Identified_Customer_Lines      0
Unique_SKUs                    0
Store_Closed                   0
Weekday                        0
Event                          0
Season_Month_Event_Mult        0
DOW_Mult                       0
Month_Mult                     0
Event_Mult                     0
Discount_Line_Prob             0
Refund_Txn_Prob                0
Profit_Margin                 68
Discount_Rate                 68
Avg_Transaction_Value         68
Units_Per_Transaction         68
Profit_Lag_1                  30
Profit_Lag_7                 210
Net_Sales_

In [28]:
store_day["Future_7D_Gross_Profit"] = (
    store_day.groupby("Location")["Gross_Profit"]
    .transform(lambda x: x.shift(-1).rolling(7).sum())
)

In [29]:
store_day[[
    "Location",
    "Date",
    "Gross_Profit",
    "Future_7D_Gross_Profit"
]].tail(10)

,Location,Date,Gross_Profit,Future_7D_Gross_Profit
21920,Store 30 - Kansas City,2025-12-22,1573.29,15842.78
21921,Store 30 - Kansas City,2025-12-23,1500.88,16719.76
21922,Store 30 - Kansas City,2025-12-24,2502.55,15517.07
21923,Store 30 - Kansas City,2025-12-25,0.00,15128.68
21924,Store 30 - Kansas City,2025-12-26,1709.51,12637.40
21925,Store 30 - Kansas City,2025-12-27,3468.03,13790.47
21926,Store 30 - Kansas City,2025-12-28,3036.21,13619.03
21927,Store 30 - Kansas City,2025-12-29,1401.85,14104.42
21928,Store 30 - Kansas City,2025-12-30,1986.27,13355.32
21929,Store 30 - Kansas City,2025-12-31,1753.45,NaN


In [30]:
store_day.to_csv("../../data/processed/store_day.csv", index=False)